# 03 — Table Order Payments

Nettoyage de la table `order_payments` : une ligne par paiement (une commande peut en avoir plusieurs).

*Extrait de `ETL-goupe.ipynb` (commit `72541c3`).*

On commence par importer les librairies nécessaires et charger les fichiers CSV dans un dictionnaire de DataFrames. Ensuite, nous créons une copie du dictionnaire original pour conserver les données brutes.

In [1]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
df_origin = {}
for file in all_files:
    nom = file.replace("datasets/Olist/", "").replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)
    df_origin[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

### Table Order Payments

In [2]:
payments = df_dic["order_payments"]

display(resume(payments))
print("Doublons exacts :", payments.duplicated().sum())

print("Clés dupliquées :", payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum())
display(payments["payment_type"].value_counts())

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,99440,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
payment_sequential,int64,29,29,1,0,0.0
payment_type,str,5,voucher,boleto,0,0.0
payment_installments,int64,24,24,0,0,0.0
payment_value,float64,29077,13664.08,0.0,0,0.0


Doublons exacts : 0
Clés dupliquées : 0


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [3]:
print("Paiements négatifs :",
      (payments["payment_value"] < 0).sum())

print("Paiements nuls :",
      (payments["payment_value"] == 0).sum())

print("Mensualités nulles :",
      (payments["payment_installments"] == 0).sum())
      


Paiements négatifs : 0
Paiements nuls : 9
Mensualités nulles : 2


### Correction des anomalies confirmées


In [4]:
payments = payments.drop_duplicates().copy()

display(payments[
    (payments["payment_value"] <= 0)
    | (payments["payment_installments"] <= 0)
])
display(payments["payment_type"].value_counts())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [5]:
display(
    payments[payments["payment_type"] == "not_defined"]
    .merge(
        df_origin["orders"][["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
)
payments["payment_type"] = payments["payment_type"].replace(
    "not_defined", "unknown"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
1,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
2,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled


### Comparaison des données avant et après transformation


In [6]:
for nom in ["order_payments"]:

    avant = df_origin[nom]
    apres = df_dic[nom]

    comparaison = pd.DataFrame({
        "Avant": [
            len(avant),
            avant.shape[1],
            avant.duplicated().sum(),
            avant.isna().sum().sum()
        ],
        "Après": [
            len(apres),
            apres.shape[1],
            apres.duplicated().sum(),
            apres.isna().sum().sum()
        ]
    }, index=[
        "Nombre de lignes",
        "Nombre de colonnes",
        "Doublons",
        "Valeurs manquantes"
    ])

    print(nom)
    display(comparaison)

order_payments


,Avant,Après
Nombre de lignes,103886,103886
Nombre de colonnes,5,5
Doublons,0,0
Valeurs manquantes,0,0
